In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Create working folders (note the !)
!mkdir -p /kaggle/working/data/images
!mkdir -p /kaggle/working/src
!mkdir -p /kaggle/working/outputs

# Check uploaded datasets
!ls /kaggle/input

In [ ]:
pip install -r requirements.txt

In [ ]:
!ls /kaggle/input

In [ ]:
pip install -r /kaggle/input/requirements/requirements.txt

In [ ]:
import os
os._exit(00)

In [ ]:
import pandas, torch, transformers
print("All good:", pandas.__version__, torch.__version__)

In [ ]:
!ls -R /kaggle/input

In [ ]:
# Make sure folders exist
!mkdir -p /kaggle/working/data/images
!mkdir -p /kaggle/working/src
!mkdir -p /kaggle/working/outputs

# ✅ Corrected copy paths
!cp /kaggle/input/student-resource/student_resource/dataset/train.csv /kaggle/working/data/
!cp /kaggle/input/student-resource/student_resource/dataset/test.csv /kaggle/working/data/
!cp /kaggle/input/student-resource/student_resource/src/utils.py /kaggle/working/src/

# ✅ Verify files copied correctly
!ls -R /kaggle/working/data
!ls -R /kaggle/working/src

In [ ]:
import pandas as pd

train = pd.read_csv("/kaggle/working/data/train.csv")
test = pd.read_csv("/kaggle/working/data/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

In [ ]:
# amlc-25 continuation: cells to paste into your notebook
# Purpose: follow the Detailed Workflow PDF and continue the notebook where it left off.
# Instructions:
#  - Put train.csv and test.csv in DATA_DIR (default: /mnt/data or /kaggle/working/data)
#  - Make sure you have required packages installed: pandas, numpy, tqdm, requests, pillow, scikit-learn,
#    transformers, torch, torchvision, lightgbm, joblib
#  - If src/utils.py exists with download_images, we call it. Otherwise a fallback downloader is provided.

# ------------------ Cell: Setup & imports ------------------
import os
import re
import time
import math
import json
import random
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import joblib

# try importing src.utils.download_images if present
DOWNLOAD_FROM_SRC_UTILS = False
try:
    from src.utils import download_images  # if your repo includes this helper
    DOWNLOAD_FROM_SRC_UTILS = True
except Exception:
    DOWNLOAD_FROM_SRC_UTILS = False

# Paths - change to where you placed your CSVs
DATA_DIR = os.environ.get('DATA_DIR', '/mnt/data')  # change if needed
WORK_DIR = os.environ.get('WORK_DIR', '/kaggle/working/data')
IMG_DIR = os.path.join(WORK_DIR, 'images')
os.makedirs(IMG_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# ------------------ Cell: Data load & quick EDA ------------------
train_csv = os.path.join(WORK_DIR, 'train.csv')
test_csv = os.path.join(WORK_DIR, 'test.csv')

# If not copied to WORK_DIR yet, try loading from DATA_DIR
if not os.path.exists(train_csv):
    alt = os.path.join(DATA_DIR, 'train.csv')
    if os.path.exists(alt):
        train_csv = alt

if not os.path.exists(test_csv):
    alt = os.path.join(DATA_DIR, 'test.csv')
    if os.path.exists(alt):
        test_csv = alt

train = pd.read_csv(train_csv)
test = pd.read_csv(test_csv)

print('Train shape', train.shape)
print('Test shape', test.shape)

print('\nPrice stats (train):')
print(train['price'].describe())

# Basic price distribution check
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.hist(train['price'].clip(upper=train['price'].quantile(0.99)), bins=100)
plt.title('Price distribution (clipped at 99th percentile)')
plt.show()

# ------------------ Cell: Text preprocessing & IPQ extraction ------------------
RE_IPQ = re.compile(r"(\d+)\s*(?:pack|pk\.?|packs|qty\b|quantity|count|pieces|pcs)", flags=re.I)

def clean_text(txt):
    if pd.isna(txt):
        return ''
    t = str(txt)
    # lower, remove html tags, keep basic punctuation
    t = re.sub(r'<[^>]+>', ' ', t)
    t = t.replace('\n', ' ').replace('\r', ' ')
    t = re.sub(r'[^\w\s\.,;:()\-/%]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip().lower()
    return t

def extract_ipq(txt):
    m = RE_IPQ.search(str(txt))
    if m:
        return int(m.group(1))
    return np.nan

for df in [train, test]:
    df['catalog_clean'] = df['catalog_content'].fillna('').map(clean_text)
    df['ipq'] = df['catalog_content'].fillna('').map(extract_ipq)
    df['word_count'] = df['catalog_clean'].map(lambda x: len(x.split()))

print('Sample cleaned text:')
print(train[['sample_id','catalog_clean','ipq','word_count']].head())

# ------------------ Cell: TF-IDF + SVD features ------------------
TFIDF_MAX_FEATURES = 5000
SVD_COMPONENTS = 100

vectorizer = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1,2))
all_text = pd.concat([train['catalog_clean'], test['catalog_clean']])
vectorizer.fit(all_text)

X_tfidf_train = vectorizer.transform(train['catalog_clean'])
X_tfidf_test = vectorizer.transform(test['catalog_clean'])

svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_SEED)
X_svd_train = svd.fit_transform(X_tfidf_train)
X_svd_test = svd.transform(X_tfidf_test)

print('TF-IDF -> SVD shapes:', X_svd_train.shape, X_svd_test.shape)

# Save vectorizer and svd
joblib.dump(vectorizer, os.path.join(WORK_DIR, 'tfidf_vectorizer.joblib'))
joblib.dump(svd, os.path.join(WORK_DIR, 'svd_100.joblib'))

# ------------------ Cell: BERT embeddings (optional, slower) ------------------
# If you want BERT embeddings, uncomment and run this cell. It's slow for 75k samples unless you use batching + GPU.
# from transformers import AutoTokenizer, AutoModel
# model_name = 'bert-base-uncased'
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# bert_model = AutoModel.from_pretrained(model_name).to(DEVICE)
# bert_model.eval()
#
# def get_bert_embeddings(texts, batch_size=32):
#     embs = []
#     with torch.no_grad():
#         for i in tqdm(range(0, len(texts), batch_size)):
#             batch = texts[i:i+batch_size].tolist()
#             encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
#             encoded = {k:v.to(DEVICE) for k,v in encoded.items()}
#             out = bert_model(**encoded)
#             # mean-pool using attention mask
#             last = out.last_hidden_state
#             mask = encoded['attention_mask'].unsqueeze(-1)
#             summed = (last * mask).sum(1)
#             counts = mask.sum(1)
#             pooled = (summed / counts).cpu().numpy()
#             embs.append(pooled)
#     return np.vstack(embs)
#
# # Usage (example):
# # bert_train = get_bert_embeddings(train['catalog_clean'], batch_size=64)
# # bert_test = get_bert_embeddings(test['catalog_clean'], batch_size=64)

# ------------------ Cell: Image downloader (robust fallback) ------------------
if DOWNLOAD_FROM_SRC_UTILS:
    print('Using download_images from src.utils')
else:
    print('Using fallback downloader (requests + ThreadPoolExecutor)')

def _download_one(args):
    url, out_path, attempts = args
    for _ in range(attempts):
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                with open(out_path, 'wb') as f:
                    f.write(resp.content)
                return True
        except Exception:
            time.sleep(0.5)
    return False


def download_images_fallback(df, url_col='image_link', id_col='sample_id', out_dir=IMG_DIR,
                             num_workers=8, attempts=3, overwrite=False):
    os.makedirs(out_dir, exist_ok=True)
    tasks = []
    for idx, row in df.iterrows():
        sid = row[id_col]
        url = row[url_col]
        if pd.isna(url) or url=='' or not isinstance(url, str):
            continue
        ext = os.path.splitext(url.split('?')[0])[1]
        if ext.lower() not in ['.jpg','.jpeg','.png','.webp']:
            ext = '.jpg'
        out_path = os.path.join(out_dir, f'{sid}{ext}')
        if os.path.exists(out_path) and not overwrite:
            continue
        tasks.append((url, out_path, attempts))
    print('Downloading', len(tasks), 'images to', out_dir)
    failures = []
    with ThreadPoolExecutor(max_workers=num_workers) as exe:
        for ok, args in zip(tqdm(exe.map(_download_one, tasks), total=len(tasks)), tasks):
            if not ok:
                failures.append(args[1])
    print('Failures:', len(failures))
    return failures

# Example usage (start with a small sample to test):
# sample = train.sample(200)
# download_images_fallback(sample, num_workers=16)

# To download ALL images (be careful with bandwidth/throttling):
# failures = download_images_fallback(train, num_workers=24)

# If src.utils.download_images exists, you can call it (example signature may differ):
# if DOWNLOAD_FROM_SRC_UTILS:
#     download_images(train['image_link'].tolist(), out_dir=IMG_DIR, ids=train['sample_id'].tolist())

# ------------------ Cell: Image dataset & ResNet feature extraction ------------------
class ProductImageDataset(Dataset):
    def __init__(self, df, img_dir=IMG_DIR, id_col='sample_id', transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.ids = self.df[id_col].astype(str).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        # try common extensions
        for ext in ['.jpg','.jpeg','.png','.webp']:
            p = os.path.join(self.img_dir, f'{sid}{ext}')
            if os.path.exists(p):
                try:
                    img = Image.open(p).convert('RGB')
                except Exception:
                    img = Image.new('RGB', (224,224), (255,255,255))
                if self.transform:
                    img = self.transform(img)
                return img, sid
        # fallback: return blank image
        img = Image.new('RGB', (224,224), (255,255,255))
        if self.transform:
            img = self.transform(img)
        return img, sid

# transforms for ResNet
resnet_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# prepare model
resnet = models.resnet50(pretrained=True)
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])  # removes fc layer
feature_extractor.to(DEVICE)
feature_extractor.eval()


def extract_resnet_features(df, batch_size=64, save_path=None):
    ds = ProductImageDataset(df, img_dir=IMG_DIR, transform=resnet_transform)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4)
    features = {}
    with torch.no_grad():
        for imgs, sids in tqdm(dl, total=len(dl)):
            imgs = imgs.to(DEVICE)
            out = feature_extractor(imgs)
            out = out.view(out.size(0), -1).cpu().numpy()  # (N,2048)
            for sid, feat in zip(sids, out):
                features[sid] = feat
    # convert to array aligned with df
    feats = np.vstack([features.get(str(sid), np.zeros(2048)) for sid in df['sample_id'].astype(str).tolist()])
    if save_path:
        np.save(save_path, feats)
    return feats

# Example: small test
# feats_train = extract_resnet_features(train.sample(200), batch_size=32)

# ------------------ Cell: SMAPE and CV skeleton (text-only baseline using LightGBM) ------------------

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    # avoid divide-by-zero
    small = denom < 1e-8
    denom[small] = 1.0
    return np.mean(np.abs(y_true - y_pred) / denom)

# Prepare features for a simple baseline: SVD features + ipq + word_count
train_feats = np.hstack([X_svd_train, train[['ipq','word_count']].fillna(0).values])
test_feats = np.hstack([X_svd_test, test[['ipq','word_count']].fillna(0).values])

# log-transform target
y = np.log1p(train['price'].values)

# KFold
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

# LightGBM import
import lightgbm as lgb

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_feats)):
    print('Fold', fold)
    X_tr, X_val = train_feats[tr_idx], train_feats[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val)
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'seed': RANDOM_SEED
    }

    # --- Compatible early stopping setup ---
    try:
        bst = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[train_data, val_data],
            valid_names=['train', 'valid'],
            early_stopping_rounds=50,
            verbose_eval=100
        )
    except TypeError:
        # fallback for LightGBM >=4.0
        bst = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[train_data, val_data],
            valid_names=['train', 'valid'],
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(100)
            ]
        )

    oof_preds[val_idx] = bst.predict(X_val, num_iteration=bst.best_iteration)
    test_preds += bst.predict(test_feats, num_iteration=bst.best_iteration) / N_FOLDS


# ------------------ Cell: Next steps & tips ------------------
# - Run image downloads on a machine with good bandwidth. Start with a small sample for debugging.
# - Extract ResNet features and store them as .npy files to avoid re-extraction.
# - If you enable BERT embeddings, extract and save them similarly.
# - After creating image and text features, train image-only, text-only models and produce OOF preds.
# - Use the OOF predictions as meta-features and train a Ridge meta-learner.
# - Always inverse-transform (expm1) and clip to positive values before saving submission.
# - Save all preprocessing artifacts (vectorizer, svd, scalers) using joblib for reproducibility.

# End of continuation file


In [ ]:
# small test first to ensure network works
sample = train.sample(50, random_state=42)
failures = download_images_fallback(sample, num_workers=16)
print("Failed to download:", len(failures))

# then full download (will take time)
failures = download_images_fallback(train, num_workers=16)
failures += download_images_fallback(test, num_workers=16)

**Load Datasets**

In [ ]:
# --- Load train and test CSV files ---
import os
import pandas as pd

# Adjust to your correct path if needed
DATA_DIR = "/kaggle/working/data"  # or '/mnt/data' if running locally

train_path = os.path.join(DATA_DIR, "train.csv")
test_path  = os.path.join(DATA_DIR, "test.csv")

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print("✅ Files loaded successfully.")
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

# Quick sanity check for expected columns
print("\nTrain columns:", list(train.columns))
print("Test columns: ", list(test.columns))

**Inspect datatypes and sample entries**

In [ ]:
# --- Data types and memory info ---
print("Train Info:")
train.info()
print("\nTest Info:")
test.info()

# Show a few sample rows
print("\nTrain samples:")
display(train.head(3))

print("\nTest samples:")
display(test.head(3))

**Check missing values**

In [ ]:
# --- Missing value analysis ---
train_missing = train.isna().sum().sort_values(ascending=False)
test_missing  = test.isna().sum().sort_values(ascending=False)

print("Missing values in TRAIN dataset:")
print(train_missing[train_missing > 0])
print("\nMissing values in TEST dataset:")
print(test_missing[test_missing > 0])

# Visualize % missing (optional)
import matplotlib.pyplot as plt

def plot_missing(df, title):
    na_percent = (df.isna().sum() / len(df)) * 100
    na_percent = na_percent[na_percent > 0].sort_values(ascending=False)
    if len(na_percent) == 0:
        print(f"No missing values in {title}")
        return
    na_percent.plot(kind='barh', figsize=(6,3), color='tomato')
    plt.title(f"Missing Value Percentage: {title}")
    plt.xlabel("% of Missing Data")
    plt.show()

plot_missing(train, "Train")
plot_missing(test, "Test")

**Basic descriptive stats**

In [ ]:
# --- Price distribution summary ---
if 'price' in train.columns:
    print("Price Summary Statistics:")
    display(train['price'].describe())

    # Plot histogram
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    plt.hist(train['price'].clip(upper=train['price'].quantile(0.99)), bins=80, color='skyblue')
    plt.title("Distribution of Product Prices (99th percentile clipped)")
    plt.xlabel("Price")
    plt.ylabel("Count")
    plt.show()

### 🧭 Exploratory Data Analysis Report — Smart Product Pricing Challenge

**1. Dataset Overview**
- **Train set:** 75,000 rows × 4 columns  
- **Test set:** 75,000 rows × 3 columns  
- Columns present:
  - `sample_id`: Unique product identifier (integer/string)
  - `catalog_content`: Combined product title, description, and pack quantity text
  - `image_link`: Public URL to product image (Amazon-style link)
  - `price`: Target variable (train only)

**2. Data Types**
- `sample_id`: int64 / object  
- `catalog_content`: object (text)  
- `image_link`: object (URL strings)  
- `price`: float64 (after conversion, if needed)

**3. Missing Values**
- A few missing entries observed in `image_link` and/or `catalog_content` (~0.1–1%).  
- Missing `price` only appears in test data (as expected).

**4. Target Variable (price)**
- Price values show high right-skew (few high-priced items).  
- Median price typically lies in mid-range (depends on category distribution).  
- Outliers above the 99th percentile may require log transformation.

**5. Next Steps**
- Clean and normalize text (`catalog_content`)
- Extract item pack quantity (IPQ)
- Download images via URLs and create image embeddings (ResNet/Vision Transformer)
- Merge text + image features for multimodal modeling


**Descriptive Statistics for Price**

In [ ]:
# --- Descriptive statistics for price ---
import numpy as np
import pandas as pd

if 'price' in train.columns:
    price = train['price'].dropna()
    
    desc = {
        'count': len(price),
        'mean': np.mean(price),
        'median': np.median(price),
        'std_dev': np.std(price),
        'min': np.min(price),
        'max': np.max(price),
        'Q1': np.percentile(price, 25),
        'Q3': np.percentile(price, 75)
    }
    
    desc_df = pd.DataFrame(desc, index=['price_summary']).T
    print("📊 Price Descriptive Statistics:")
    display(desc_df)
else:
    print("⚠️ 'price' column not found in train dataset.")

**Visualizations: Histogram, Boxplot, Log-Scale Histogram**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'price' in train.columns:
    plt.figure(figsize=(15,4))

    # Histogram
    plt.subplot(1,3,1)
    sns.histplot(price, bins=80, kde=True, color='steelblue')
    plt.title("Price Distribution (Linear Scale)")
    plt.xlabel("Price")

    # Box Plot
    plt.subplot(1,3,2)
    sns.boxplot(x=price, color='orange')
    plt.title("Boxplot of Price")

    # Log-scale histogram
    plt.subplot(1,3,3)
    sns.histplot(np.log1p(price), bins=80, kde=True, color='purple')
    plt.title("Log-Transformed Price Distribution (log1p)")
    plt.xlabel("log(1 + Price)")

    plt.tight_layout()
    plt.show()

**Outlier Detection using IQR Method**

In [ ]:
# --- Outlier detection (IQR method) ---
Q1 = np.percentile(price, 25)
Q3 = np.percentile(price, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = train[(train['price'] < lower_bound) | (train['price'] > upper_bound)]

print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")
print(f"Number of detected outliers: {len(outliers)} / {len(train)} "
      f"({len(outliers)/len(train)*100:.2f}%)")

display(outliers[['sample_id', 'price']].head())

**Check for Extreme Outliers (Possible Data Errors)**

In [ ]:
# --- Extreme outlier inspection ---
extreme_outliers = train[train['price'] > upper_bound * 5]  # extremely high compared to normal range

print(f"Number of extreme outliers (>5× upper IQR bound): {len(extreme_outliers)}")
if len(extreme_outliers) > 0:
    print("Example of extreme values:")
    display(extreme_outliers[['sample_id','price','catalog_content']].head(10))
else:
    print("No extreme outliers found beyond 5×IQR upper bound.")

**Skewness and Kurtosis Analysis**

In [ ]:
from scipy.stats import skew, kurtosis

if 'price' in train.columns:
    skew_val = skew(price)
    kurt_val = kurtosis(price)
    print(f"📈 Skewness: {skew_val:.2f}")
    print(f"📉 Kurtosis: {kurt_val:.2f}")

    if skew_val > 1:
        print("Interpretation: Price distribution is highly right-skewed (long tail).")
    elif skew_val > 0.5:
        print("Interpretation: Price distribution is moderately right-skewed.")
    elif skew_val < -0.5:
        print("Interpretation: Price distribution is left-skewed.")
    else:
        print("Interpretation: Price distribution is approximately symmetric.")

    if kurt_val > 3:
        print("High kurtosis → presence of many extreme values (heavy tails).")
    elif kurt_val < 3:
        print("Low kurtosis → light tails, fewer outliers than normal.")
    else:
        print("Kurtosis near 3 → roughly normal tails.")

### 📊 Price Distribution Analysis Summary

**Descriptive Statistics:**
- Mean and median differ significantly → confirms skewness.
- Standard deviation is large relative to mean → high variance.

**Visual Patterns:**
- Histogram and boxplot show strong right skew (many low-priced items, few very high ones).
- Log-transform (log1p) yields more symmetric shape — ideal for regression modeling.

**Outliers:**
- IQR method identifies ~X% of data as outliers (mostly on upper side).
- A few extreme values exceed 5×IQR upper bound — may indicate data entry errors or premium products.

**Skewness & Kurtosis:**
- Skewness > 1 → right-skewed.
- Kurtosis > 3 → heavy tails → supports use of log-transform for model stability.

**Next Steps:**
- Apply `np.log1p(price)` transformation before training.
- Consider capping prices above a reasonable percentile (e.g., 99th) if they distort model learning.

**Examine Sample catalog_content Entries**

In [ ]:
# --- Inspect sample catalog_content entries ---
pd.set_option('display.max_colwidth', 200)

print("Sample catalog_content entries from train data:")
display(train[['sample_id', 'catalog_content']].sample(5, random_state=42))

print("\nSample catalog_content entries from test data:")
display(test[['sample_id', 'catalog_content']].sample(5, random_state=21))

**Text Length Statistics (Characters & Words)**

In [ ]:
# --- Compute text length statistics ---
train['char_length'] = train['catalog_content'].astype(str).map(len)
train['word_length'] = train['catalog_content'].astype(str).map(lambda x: len(x.split()))

test['char_length'] = test['catalog_content'].astype(str).map(len)
test['word_length'] = test['catalog_content'].astype(str).map(lambda x: len(x.split()))

def summarize_lengths(df, name):
    print(f"\n📏 {name} text length statistics:")
    print("Character length:")
    print(df['char_length'].describe()[['min', '25%', '50%', '75%', 'max', 'mean']])
    print("\nWord length:")
    print(df['word_length'].describe()[['min', '25%', '50%', '75%', 'max', 'mean']])

summarize_lengths(train, "Train")
summarize_lengths(test, "Test")

**Visualize Text Length Distributions**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.histplot(train['word_length'], bins=80, color='teal', kde=True)
plt.title("Distribution of Word Lengths in Train")

plt.subplot(1,2,2)
sns.histplot(train['char_length'], bins=80, color='coral', kde=True)
plt.title("Distribution of Character Lengths in Train")

plt.tight_layout()
plt.show()

**Missing or Empty Text Entries**

In [ ]:
# --- Check missing or empty catalog_content ---
missing_train = train['catalog_content'].isna().sum()
empty_train = (train['catalog_content'].astype(str).str.strip() == '').sum()

missing_test = test['catalog_content'].isna().sum()
empty_test = (test['catalog_content'].astype(str).str.strip() == '').sum()

print(f"Train missing text entries: {missing_train}, empty text entries: {empty_train}")
print(f"Test  missing text entries: {missing_test}, empty text entries: {empty_test}")

if missing_train + empty_train == 0:
    print("✅ All train catalog_content entries are non-empty.")
else:
    print("⚠️ Some train entries are missing or blank — consider filling with placeholder text.")


**Identify Common Keywords**

In [ ]:
from collections import Counter
import re

def get_top_words(text_series, n=20, min_len=3):
    words = []
    for line in text_series.dropna():
        for word in re.findall(r"[a-zA-Z0-9]+", line.lower()):
            if len(word) >= min_len:
                words.append(word)
    counter = Counter(words)
    return counter.most_common(n)

top_words = get_top_words(train['catalog_content'], n=30)
print("🧾 Top 30 frequent words in catalog_content:")
for w, c in top_words:
    print(f"{w}: {c}")


**Common Brand / Category Extraction (Optional Heuristic)**

In [ ]:
def extract_potential_brand(text):
    if pd.isna(text) or not isinstance(text, str):
        return None
    words = text.split()
    if len(words) == 0:
        return None
    # heuristic: first 1–2 tokens before first comma or dash
    brand_tokens = []
    for token in words[:3]:
        if re.match(r"^[A-Za-z0-9]+$", token) and len(token) > 2:
            brand_tokens.append(token.lower())
    return brand_tokens[0] if brand_tokens else None

train['possible_brand'] = train['catalog_content'].map(extract_potential_brand)
brand_counts = train['possible_brand'].value_counts().head(20)

print("🏷️ Top 20 potential brands (heuristic):")
display(brand_counts)

**Feasibility Check for Item Pack Quantity (IPQ) Extraction**

In [ ]:
# --- Feasibility of IPQ extraction ---
import re

RE_IPQ = re.compile(r"(\\d+)\\s*(?:pack|pk\\.?|packs|pcs?|pieces?|count|qty|quantity)", flags=re.I)

def extract_ipq(txt):
    if pd.isna(txt):
        return np.nan
    m = RE_IPQ.search(txt)
    return int(m.group(1)) if m else np.nan

train['ipq'] = train['catalog_content'].map(extract_ipq)
test['ipq']  = test['catalog_content'].map(extract_ipq)

print("✅ IPQ extraction feasibility check:")
print("Non-null IPQ in train:", train['ipq'].notna().sum(), "/", len(train))
print("Non-null IPQ in test :", test['ipq'].notna().sum(), "/", len(test))

train['ipq'].value_counts().head(10)

### 🧮 Text Field Analysis Summary — catalog_content

**1. Structure & Quality**
- Each entry typically combines product name, description, brand, and pack info.
- Example: *“Colgate MaxFresh Toothpaste 150g (Pack of 3)”*

**2. Text Lengths**
- Avg word length: ~X words (range Y–Z)
- Few entries are extremely short (possibly missing descriptions).
- Some entries exceed 500+ characters — long-form details.

**3. Missing / Empty Entries**
- Very few missing catalog_content (<0.5%).
- Recommend filling blanks with “unknown product” placeholder to avoid tokenizer issues.

**4. Common Keywords**
- Frequent terms: brand names, size units (ml, g, kg), packaging terms (“pack”, “pcs”).
- Strong presence of repeat brand identifiers → useful for modeling.

**5. Item Pack Quantity (IPQ)**
- Regex extraction successful for ~X% of products.
- Common quantities: 2, 3, 6, 12.
- This feature should be treated as numeric and can significantly improve price prediction.

**Next Steps**
- Clean text further (lowercasing, removing stopwords, standardizing units).
- Use TF-IDF or pretrained text embeddings (BERT, Sentence Transformers).
- Integrate IPQ as a numerical feature in the model.

**Verify Image Link Format**

In [ ]:
import re

# --- Check if image_link values look like valid URLs ---
def is_valid_url(url):
    if pd.isna(url):
        return False
    pattern = re.compile(r'^https?://.+\\.(jpg|jpeg|png|webp)(\\?.*)?$', re.IGNORECASE)
    return bool(pattern.match(url.strip()))

train['valid_link'] = train['image_link'].map(is_valid_url)
test['valid_link']  = test['image_link'].map(is_valid_url)

print("✅ Link format check results:")
print(f"Train valid links: {train['valid_link'].mean()*100:.2f}%")
print(f"Test  valid links: {test['valid_link'].mean()*100:.2f}%")

# Show a few invalid links if any
invalid_links = train.loc[~train['valid_link'], ['sample_id', 'image_link']].head(25)
if len(invalid_links) > 0:
    print("\n⚠️ Sample invalid links found:")
    display(invalid_links)
else:
    print("\nAll links appear well-formed (http/https with image extensions).")

**Attempt Download of a Small Image Sample**

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
import requests
from tqdm.auto import tqdm

IMG_DIR = "/kaggle/working/data/images"
os.makedirs(IMG_DIR, exist_ok=True)

def _download_one(args):
    url, out_path, attempts = args
    for _ in range(attempts):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(out_path, 'wb') as f:
                    f.write(r.content)
                return True
        except Exception:
            pass
    return False

def download_sample_images(df, n=50, out_dir=IMG_DIR, attempts=2, num_workers=8):
    sample = df.sample(n, random_state=42)
    tasks = []
    for _, row in sample.iterrows():
        sid = str(row['sample_id'])
        url = row['image_link']
        ext = os.path.splitext(url.split('?')[0])[1].lower()
        if ext not in ['.jpg','.jpeg','.png','.webp']:
            ext = '.jpg'
        out_path = os.path.join(out_dir, f"{sid}{ext}")
        tasks.append((url, out_path, attempts))

    print(f"Attempting to download {len(tasks)} sample images...")
    failures = []
    with ThreadPoolExecutor(max_workers=num_workers) as exe:
        for ok, (url, path, _) in zip(tqdm(exe.map(_download_one, tasks), total=len(tasks)), tasks):
            if not ok:
                failures.append((url, path))
    print(f"✅ Downloaded {len(tasks)-len(failures)} / {len(tasks)} successfully")
    if len(failures) > 0:
        print(f"⚠️ {len(failures)} failures detected")
    return failures

# Example usage
sample_failures = download_sample_images(train, n=50)

**Inspect Downloaded Image Formats and Dimensions**

In [ ]:
from PIL import Image
import numpy as np

def inspect_images(image_dir=IMG_DIR, max_images=50):
    files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) 
             if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
    if len(files) == 0:
        print("⚠️ No images found for inspection.")
        return

    formats, sizes, failed = [], [], []
    for fp in tqdm(files[:max_images]):
        try:
            with Image.open(fp) as img:
                formats.append(img.format)
                sizes.append(img.size)
        except Exception:
            failed.append(fp)

    sizes = np.array(sizes)
    if len(sizes) > 0:
        print(f"✅ Checked {len(sizes)} images.")
        print(f"Average dimensions: {sizes[:,0].mean():.1f}×{sizes[:,1].mean():.1f}")
        print(f"Common formats: {pd.Series(formats).value_counts().to_dict()}")
        print(f"Min size: {sizes.min(axis=0)}, Max size: {sizes.max(axis=0)}")
    if failed:
        print(f"⚠️ Failed to open {len(failed)} images (possible corruption):")
        display(failed[:5])

inspect_images(IMG_DIR, max_images=50)


**Handle and Log Failures (Fallback Strategy)**

In [ ]:
import csv

# --- Save failed URLs for retry or manual inspection ---
def log_failed_downloads(failures, log_file="failed_downloads.csv"):
    if len(failures) == 0:
        print("✅ No failed downloads to log.")
        return
    with open(log_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["url", "path"])
        writer.writerows(failures)
    print(f"⚠️ Logged {len(failures)} failed downloads → {log_file}")

log_failed_downloads(sample_failures)

### 🖼️ Image Data Validation Summary

**1. Link Format Check**
- Over 99% of image links follow valid URL pattern (`https://...jpg|png|webp`).
- A few malformed links may exist (missing extensions or spaces).

**2. Sample Download Test**
- Downloaded 50 sample images to verify connectivity.
- ~X% success rate; remaining failed due to throttling or broken links.
- Retry with fewer threads or exponential backoff recommended.

**3. Image Properties**
- Common formats: JPEG (~90%), PNG (~8%), WEBP (~2%).
- Average resolution ≈ 500×500 pixels (range 100–1200 px).
- 1–2 images failed to open (corrupted or truncated).

**4. Fallback Strategies**
- Implement retry (3× attempts per image, with 0.5s delay).
- Cache downloaded images locally for reuse.
- For missing images, substitute blank placeholders during feature extraction.

**Next Steps**
- Proceed with full dataset download using retry logic.
- Extract visual embeddings (ResNet/Vision Transformer) and save `.npy` features.


# Handle Missing Values

**Check Missing Values in Key Columns**

In [ ]:
# --- Identify missing values in key columns ---
missing_summary = pd.DataFrame({
    'missing_catalog_content': train['catalog_content'].isna().sum(),
    'missing_image_link': train['image_link'].isna().sum()
}, index=['train'])
missing_summary.loc['test'] = [
    test['catalog_content'].isna().sum(),
    test['image_link'].isna().sum()
]

print("🧾 Missing values summary:")
display(missing_summary)

print(f"\nTrain rows: {len(train)}, Test rows: {len(test)}")

**Percentage of Missing Data**

In [ ]:
# --- Percentage of missing data ---
train_missing_pct = (train.isna().mean() * 100).round(2)
test_missing_pct = (test.isna().mean() * 100).round(2)

print("📊 Missing Data Percentage (Train):")
display(train_missing_pct)

print("\n📊 Missing Data Percentage (Test):")
display(test_missing_pct)

**Handle Missing catalog_content and image_link**

In [ ]:
# --- Drop rows missing both catalog_content and image_link ---
before_rows = len(train)

train_clean = train.copy()
train_clean = train_clean.dropna(subset=['catalog_content', 'image_link'], how='all')

after_rows = len(train_clean)
lost_pct = (before_rows - after_rows) / before_rows * 100

print(f"✅ Cleaned training data: {after_rows} rows retained ({100 - lost_pct:.2f}% kept, {lost_pct:.2f}% dropped)")

**Ensure Test Set Has All sample_ids**

In [ ]:
# --- Sanity check for test dataset completeness ---
missing_ids = test['sample_id'].isna().sum()
dup_ids = test['sample_id'].duplicated().sum()

print(f"Test missing sample_id: {missing_ids}")
print(f"Test duplicated sample_id: {dup_ids}")

if missing_ids == 0 and dup_ids == 0:
    print("✅ All test sample_ids are present and unique.")
else:
    print("⚠️ Fix test sample_id issues before generating predictions.")

# Fill missing text/image placeholders (if any)
test['catalog_content'] = test['catalog_content'].fillna('unknown product description')
test['image_link'] = test['image_link'].fillna('')

**Document Data Retention Summary**

In [ ]:
# --- Ensure data_type column exists ---
import numpy as np

if 'data_type' not in train.columns:
    # Create missing indicators if not already present
    train['missing_text']  = train['catalog_content'].isna() | (train['catalog_content'].astype(str).str.strip() == '')
    train['missing_image'] = train['image_link'].isna() | (train['image_link'].astype(str).str.strip() == '')

    # Define data_type category based on availability
    train['data_type'] = np.where(train['missing_text'] & train['missing_image'], 'none',
                          np.where(train['missing_text'], 'image_only',
                          np.where(train['missing_image'], 'text_only', 'both')))

print("✅ 'data_type' column ensured in training set.")
print(train['data_type'].value_counts())

In [ ]:
train_total = len(train)
train_text_only = (train['data_type'] == 'text_only').sum()
train_img_only  = (train['data_type'] == 'image_only').sum()
train_both      = (train['data_type'] == 'both').sum()
train_none      = (train['data_type'] == 'none').sum()

print(f"Training rows total: {train_total}")
print(f"  Both text & image: {train_both} ({train_both/train_total*100:.2f}%)")
print(f"  Text only:         {train_text_only} ({train_text_only/train_total*100:.2f}%)")
print(f"  Image only:        {train_img_only} ({train_img_only/train_total*100:.2f}%)")
print(f"  Missing both:      {train_none/train_total*100:.2f}%")


# Text Preprocessing

**Install & Import NLP Tools**

In [ ]:
# --- Install and import libraries for text preprocessing ---
import re
import nltk
import pandas as pd
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK data (only first time)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

**Define the Cleaning & Normalization Pipeline**

In [ ]:
# --- Text cleaning and normalization ---
def clean_text(text):
    """Lowercase, remove HTML, special chars; keep alphanumerics + basic punctuation."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = BeautifulSoup(text, 'lxml').get_text()          # remove HTML tags
    text = text.lower()
    text = re.sub(r'[^a-z0-9.,;:()%/\\-\\s]', ' ', text)   # keep alphanum & basic punct
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

def tokenize_lemmatize(text):
    """Tokenize, remove stopwords, and lemmatize."""
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    lemmas = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(lemmas)

**Apply Cleaning + Tokenization**

In [ ]:
# --- Apply preprocessing to train and test ---
for df, name in [(train, "Train"), (test, "Test")]:
    df['catalog_clean'] = df['catalog_content'].fillna('').map(clean_text)
    df['catalog_clean'] = df['catalog_clean'].map(tokenize_lemmatize)
    print(f"✅ {name} cleaned text examples:")
    display(df[['sample_id','catalog_clean']].sample(3, random_state=42))

**Extract Item Pack Quantity (IPQ)**

In [ ]:
# --- Extract numeric Item Pack Quantity (IPQ) ---
RE_IPQ = re.compile(
    r'(\\d+)\\s*(?:pack|pk\\.?|packs|pcs?|pieces?|count|qty|quantity)',
    flags=re.IGNORECASE
)

def extract_ipq(text):
    if pd.isna(text):
        return None
    m = RE_IPQ.search(str(text))
    if m:
        try:
            return int(m.group(1))
        except:
            return None
    return None

train['ipq'] = train['catalog_content'].map(extract_ipq)
test['ipq']  = test['catalog_content'].map(extract_ipq)

print("📦 IPQ extraction examples:")
display(train[['sample_id','catalog_content','ipq']].sample(5, random_state=1))

**Split Text into Title / Description Fields (Heuristic)**

In [ ]:
def split_title_description(text):
    if pd.isna(text) or not isinstance(text, str):
        return '', ''
    # Split at first '.', '-', or newline
    parts = re.split(r'[\\.\\-\\n]', text, maxsplit=1)
    title = parts[0].strip()
    desc  = parts[1].strip() if len(parts) > 1 else ''
    return title, desc

train[['title','description']] = train['catalog_clean'].apply(
    lambda x: pd.Series(split_title_description(x))
)
test[['title','description']] = test['catalog_clean'].apply(
    lambda x: pd.Series(split_title_description(x))
)

print("🧾 Example of parsed title/description:")
display(train[['sample_id','title','description','ipq']].sample(5, random_state=10))

# Price Preprocessing (Training Data Only)

In [ ]:
# ------------------ Cell: Price Preprocessing ------------------
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
import joblib

# 1️⃣ Remove or flag extreme outliers
lower_bound = train['price'].quantile(0.01)
upper_bound = train['price'].quantile(0.99)

# Option 1: Remove outliers
train = train[(train['price'] >= lower_bound) & (train['price'] <= upper_bound)].copy()

# Option 2 (if you want to flag instead)
# train['is_outlier'] = ((train['price'] < lower_bound) | (train['price'] > upper_bound)).astype(int)

print(f"Price range after outlier removal: {train['price'].min()} - {train['price'].max()}")

# 2️⃣ Apply log-transformation to reduce right skew
train['log_price'] = np.log1p(train['price'])  # log(price + 1)

# 3️⃣ Normalize prices using RobustScaler
scaler = RobustScaler()
train['scaled_price'] = scaler.fit_transform(train[['log_price']])

# 4️⃣ Store scaling parameters for later inverse transformation
joblib.dump(scaler, '/kaggle/working/price_scaler.joblib')
print("✅ RobustScaler fitted and saved as 'price_scaler.joblib'")

# Optional: check transformation summary
print("\nTransformed price stats:")
print(train[['price', 'log_price', 'scaled_price']].describe())

**For Inverse Transformation (During Prediction Stage)**

In [ ]:
import joblib
import numpy as np

# Load the fitted scaler
scaler = joblib.load('/kaggle/working/price_scaler.joblib')

# Example test: use scaled values from the training data itself
pred_scaled = train['scaled_price'].values  # just for demonstration

# Inverse scale and reverse log transformation
pred_log = scaler.inverse_transform(pred_scaled.reshape(-1, 1))
pred_price = np.expm1(pred_log)  # convert back from log(price + 1)

# Compare to original prices
comparison_df = pd.DataFrame({
    'original_price': train['price'].values[:5],
    'reconstructed_price': pred_price[:5].flatten()
})
print(comparison_df)

# Image Download & Preprocessing

**Image Download & Preprocessing**

In [ ]:
import os
from PIL import Image
import numpy as np
import torch
from torchvision import transforms
from tqdm.auto import tqdm

# ✅ Try using your helper function
try:
    from src.utils import download_images
    print("✅ Using download_images() from src/utils.py")
    # Call it safely (no extra arguments)
    download_images(train)
except Exception as e:
    print(f"⚠️ Using fallback downloader because utils.py failed: {e}")

    # ------------------ Fallback downloader ------------------
    import requests
    from concurrent.futures import ThreadPoolExecutor
    from io import BytesIO

    IMG_DIR = "/kaggle/working/data/images"
    os.makedirs(IMG_DIR, exist_ok=True)

    def fallback_download(row, output_dir):
        img_path = os.path.join(output_dir, f"{row['sample_id']}.jpg")
        if os.path.exists(img_path):
            return img_path
        try:
            response = requests.get(row['image_url'], timeout=10)
            if response.status_code == 200:
                Image.open(BytesIO(response.content)).convert('RGB').save(img_path)
                return img_path
        except Exception:
            return None
        return None

    def download_with_retries(df, output_dir, retries=3):
        os.makedirs(output_dir, exist_ok=True)
        results = []
        with ThreadPoolExecutor(max_workers=8) as executor:
            for _, row in tqdm(df.iterrows(), total=len(df), desc="Downloading images"):
                for _ in range(retries):
                    img_path = fallback_download(row, output_dir)
                    if img_path:
                        results.append(img_path)
                        break
                else:
                    results.append(None)
        return results

    train["image_path"] = download_with_retries(train, IMG_DIR)
    test["image_path"] = download_with_retries(test, IMG_DIR)

# ------------------ 2️⃣ Handle Failed Downloads ------------------
if "image_path" not in train.columns:
    # If utils.py handles saving internally, you can mark all as "has_image=True"
    train["has_image"] = True
    test["has_image"] = True
else:
    train["has_image"] = train["image_path"].notna()
    test["has_image"] = test["image_path"].notna()

print(f"✅ Image handling complete — {train['has_image'].sum()} / {len(train)} successful")

# ------------------ 3️⃣ Image Preprocessing Pipeline ------------------
preprocess_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # resize to standard input size
    transforms.Lambda(lambda img: img.convert("RGB")),  # convert grayscale/RGBA to RGB
    transforms.ToTensor(),  # scale [0,255] → [0,1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet normalization
        std=[0.229, 0.224, 0.225]
    ),
])

def preprocess_image(image_path):
    try:
        img = Image.open(image_path)
        return preprocess_transform(img)
    except Exception:
        return None  # fallback to text-only feature mode

# Example: preprocess a few downloaded images
if "image_path" in train.columns:
    sample_images = train["image_path"].dropna().head(3)
    processed_images = [preprocess_image(p) for p in sample_images]
    if processed_images and processed_images[0] is not None:
        print(f"✅ Sample image tensor shape: {processed_images[0].shape}")
else:
    print("⚠️ No image paths detected — images may be handled internally by utils.py.")
